# 03 — Cleaning dan Preprocessing
## Persiapan Data untuk Klasifikasi Covertype

Tahap ini memeriksa kualitas data, menentukan perlakuan terhadap nilai ekstrem, melakukan transformasi fitur, seleksi fitur, dan membagi data untuk modelling. Keputusan harus didasarkan pada temuan EDA.

## 1. Import library dan memuat data

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold


from pathlib import Path
import pandas as pd
import numpy as np

NUMERIC_FEATURES = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]
WILDERNESS_COLS = [f"Wilderness_Area{i}" for i in range(1, 5)]
SOIL_COLS = [f"Soil_Type{i}" for i in range(1, 41)]
TARGET_COL = "Cover_Type"
COLUMNS = NUMERIC_FEATURES + WILDERNESS_COLS + SOIL_COLS + [TARGET_COL]

DATA_CANDIDATES = [
    Path("covtype.data"),
    Path("data") / "covtype.data",
    Path("../covtype.data"),
    Path("../data") / "covtype.data",
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "File covtype.data tidak ditemukan. Letakkan file tersebut di folder "
        "yang sama dengan notebook atau di subfolder 'data/'."
    )

df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)
print(f"Dataset berhasil dimuat dari: {DATA_PATH.resolve()}")
print(f"Ukuran dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom")

Dataset berhasil dimuat dari: C:\Users\parsey\Downloads\Covertype_4_Phase_Notebooks\covtype.data
Ukuran dataset: 581,012 baris x 55 kolom


## 2. Kondisi data sebelum cleaning

In [2]:
before_rows, before_cols = df.shape
before_missing = int(df.isna().sum().sum())
before_duplicates = int(df.duplicated().sum())

print(f"Ukuran sebelum cleaning: {before_rows:,} x {before_cols}")
print("Missing values sebelum cleaning:", before_missing)
print("Duplikasi sebelum cleaning:", before_duplicates)
display(df.head())

Ukuran sebelum cleaning: 581,012 x 55
Missing values sebelum cleaning: 0
Duplikasi sebelum cleaning: 0


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,5


## 3. Data cleaning dan validasi

### 3.1 Missing values

Pada file Covertype yang digunakan, pemeriksaan awal menemukan tidak ada missing value. Karena itu, imputasi tidak dilakukan. Jika file yang berbeda ternyata memiliki missing value, proses dihentikan agar strategi imputasi ditentukan secara sadar dan didokumentasikan, bukan dilakukan secara otomatis tanpa pemeriksaan.

In [3]:
df_clean = df.copy()

missing_by_column = df_clean.isna().sum()
if missing_by_column.sum() > 0:
    display(missing_by_column[missing_by_column > 0])
    raise ValueError(
        "Ditemukan missing value. Periksa penyebab dan tentukan strategi "
        "imputasi yang sesuai sebelum melanjutkan."
    )
else:
    print("Tidak ditemukan missing value; imputasi tidak diperlukan.")

Tidak ditemukan missing value; imputasi tidak diperlukan.


### 3.2 Duplikasi

Baris duplikat identik diperiksa. Jika ditemukan, baris duplikat dihapus dan jumlah sebelum/sesudah dicatat. Pada file yang diperiksa, tidak ditemukan duplikasi.

In [4]:
duplicates_before = int(df_clean.duplicated().sum())

if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates().copy()
    print(f"{duplicates_before:,} baris duplikat dihapus.")
else:
    print("Tidak ditemukan baris duplikat; tidak ada baris yang dihapus.")

print("Duplikasi setelah proses:", int(df_clean.duplicated().sum()))

Tidak ditemukan baris duplikat; tidak ada baris yang dihapus.
Duplikasi setelah proses: 0


### 3.3 Validasi rentang dan integritas fitur

- `Cover_Type` harus berada pada rentang 1–7.
- Kolom Wilderness Area dan Soil Type harus berisi nilai biner 0/1.
- Setiap baris seharusnya memiliki tepat satu indikator aktif pada masing-masing kelompok one-hot.
- `Vertical_Distance_To_Hydrology` dapat bernilai negatif; nilai negatif tidak otomatis salah.
- `Aspect` adalah sudut melingkar; nilai 0° dan 360° merepresentasikan arah yang sama.

In [5]:
assert df_clean[TARGET_COL].between(1, 7).all(), "Label target di luar rentang 1–7."
assert set(pd.unique(df_clean[WILDERNESS_COLS].to_numpy().ravel())).issubset({0, 1})
assert set(pd.unique(df_clean[SOIL_COLS].to_numpy().ravel())).issubset({0, 1})
assert (df_clean[WILDERNESS_COLS].sum(axis=1) == 1).all(), \
    "Ada baris dengan indikator Wilderness Area tidak tepat satu."
assert (df_clean[SOIL_COLS].sum(axis=1) == 1).all(), \
    "Ada baris dengan indikator Soil Type tidak tepat satu."

print("Validasi target dan kelompok fitur one-hot berhasil.")
print("Rentang Vertical_Distance_To_Hydrology:",
      df_clean["Vertical_Distance_To_Hydrology"].min(),
      "sampai",
      df_clean["Vertical_Distance_To_Hydrology"].max())

Validasi target dan kelompok fitur one-hot berhasil.
Rentang Vertical_Distance_To_Hydrology: -173 sampai 601


## 4. Penanganan outlier

IQR digunakan untuk menandai nilai ekstrem. Namun, nilai ekstrem pada elevasi, jarak, kemiringan, dan hillshade dapat menggambarkan kondisi geografis yang valid. Oleh karena itu, nilai outlier **dipertahankan** selama tidak terbukti sebagai kesalahan pencatatan.

Keputusan ini juga menghindari penghapusan observasi kelas minoritas yang jumlahnya sudah sedikit.

In [6]:
q1 = df_clean[NUMERIC_FEATURES].quantile(0.25)
q3 = df_clean[NUMERIC_FEATURES].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_counts = (
    (df_clean[NUMERIC_FEATURES] < lower) |
    (df_clean[NUMERIC_FEATURES] > upper)
).sum()

display(pd.DataFrame({
    "Outlier IQR": outlier_counts,
    "Persentase (%)": (outlier_counts / len(df_clean) * 100).round(2)
}).sort_values("Outlier IQR", ascending=False))

print("Outlier dipertahankan karena belum ada bukti bahwa nilai tersebut merupakan kesalahan data.")

,Outlier IQR,Persentase (%)
Vertical_Distance_To_Hydrology,31463,5.42
Horizontal_Distance_To_Fire_Points,31157,5.36
Hillshade_9am,17433,3.00
Hillshade_Noon,15672,2.70
Elevation,15569,2.68
Slope,15316,2.64
Horizontal_Distance_To_Hydrology,14557,2.51
Hillshade_3pm,7832,1.35
Horizontal_Distance_To_Roadways,669,0.12
Aspect,0,0.00


Outlier dipertahankan karena belum ada bukti bahwa nilai tersebut merupakan kesalahan data.


## 5. Feature engineering: transformasi `Aspect`

`Aspect` merupakan variabel sudut. Jika digunakan sebagai angka linear, jarak numerik antara 1° dan 359° tampak besar, padahal kedua arah tersebut berdekatan. Transformasi sinus dan cosinus merepresentasikan sifat melingkar:

\[
Aspect_{sin} = \sin(\theta), \qquad Aspect_{cos} = \cos(\theta)
\]

dengan \(\theta\) dalam radian.

In [7]:
df_model = df_clean.copy()

angle_rad = np.deg2rad(df_model["Aspect"])
df_model["Aspect_sin"] = np.sin(angle_rad)
df_model["Aspect_cos"] = np.cos(angle_rad)

# Kolom Aspect asli diganti oleh dua representasi sikliknya.
df_model = df_model.drop(columns=["Aspect"])

display(df_model[["Aspect_sin", "Aspect_cos"]].head())
print("Jumlah fitur setelah transformasi Aspect (termasuk target):", df_model.shape[1])

,Aspect_sin,Aspect_cos
0,0.777146,0.629320
1,0.829038,0.559193
2,0.656059,-0.754710
3,0.422618,-0.906308
4,0.707107,0.707107


Jumlah fitur setelah transformasi Aspect (termasuk target): 56


## 6. Pemisahan fitur dan target

`Cover_Type` adalah label yang ingin diprediksi sehingga tidak boleh dimasukkan sebagai fitur. Semua fitur kartografis lain dipertahankan untuk evaluasi awal karena masing-masing memiliki makna domain. Tidak ada fitur yang dihapus hanya karena korelasi linearnya rendah.

In [8]:
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL].astype(int)

print("Jumlah baris X:", len(X))
print("Jumlah fitur awal setelah feature engineering:", X.shape[1])
print("Jumlah kelas target:", y.nunique())

Jumlah baris X: 581012
Jumlah fitur awal setelah feature engineering: 55
Jumlah kelas target: 7


## 7. Train-test split secara stratified

Data dibagi menjadi 80% training dan 20% testing. `stratify=y` menjaga proporsi setiap kelas agar kelas minoritas tetap terwakili pada kedua bagian. `random_state=42` digunakan agar eksperimen dapat direproduksi.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

split_check = pd.DataFrame({
    "Seluruh data (%)": y.value_counts(normalize=True).sort_index() * 100,
    "Train (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Test (%)": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)
display(split_check)

X_train: (464809, 55)
X_test : (116203, 55)
y_train: (464809,)
y_test : (116203,)


,Seluruh data (%),Train (%),Test (%)
Cover_Type,,,
1,36.46,36.46,36.46
2,48.76,48.76,48.76
3,6.15,6.15,6.15
4,0.47,0.47,0.47
5,1.63,1.63,1.63
6,2.99,2.99,2.99
7,3.53,3.53,3.53


## 8. Feature selection dengan VarianceThreshold

Seleksi ini dilakukan pada data training saja untuk menghindari penggunaan informasi dari data test. Fitur dengan variansi nol (konstan) akan dihapus karena tidak memberikan informasi pembeda.

Jika tidak ada fitur yang konstan, seluruh fitur dipertahankan. Hal tersebut bukan kegagalan seleksi: fitur one-hot yang jarang tetap dapat bermakna untuk kelas tertentu.

In [10]:
selector = VarianceThreshold(threshold=0.0)
X_train_selected = selector.fit_transform(X_train)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()].tolist()
removed_features = X_train.columns[~selector.get_support()].tolist()

print("Jumlah fitur sebelum seleksi:", X_train.shape[1])
print("Jumlah fitur setelah seleksi:", len(selected_features))
print("Fitur yang dipertahankan:", selected_features)
print("Fitur yang dihapus karena variansi nol:", removed_features)

Jumlah fitur sebelum seleksi: 55
Jumlah fitur setelah seleksi: 55
Fitur yang dipertahankan: ['Elevation', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4', 'Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 'Soil_Type15', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40', 'Aspect_sin', 'Aspe

## 9. Scaling: apakah diperlukan?

Model utama yang dibandingkan pada notebook modelling adalah Decision Tree dan Random Forest. Model berbasis pohon membagi data berdasarkan ambang fitur sehingga standardisasi atau normalisasi tidak wajib. Menghindari scaling juga mempertahankan interpretasi nilai asli fitur.

Jika eksperimen tambahan menggunakan SVM atau Logistic Regression, scaling perlu dipertimbangkan. Scaler harus di-*fit* hanya pada data training, lalu digunakan untuk mentransformasi data test.

## 10. Kondisi data sesudah cleaning dan preprocessing

In [11]:
after_rows = len(df_clean)
after_missing = int(df_clean.isna().sum().sum())
after_duplicates = int(df_clean.duplicated().sum())

comparison = pd.DataFrame({
    "Sebelum": [before_rows, before_cols, before_missing, before_duplicates],
    "Sesudah cleaning": [after_rows, df_clean.shape[1], after_missing, after_duplicates]
}, index=["Jumlah baris", "Jumlah kolom", "Missing values", "Duplikasi"])
display(comparison)

print("Jumlah fitur model setelah feature engineering dan seleksi:", len(selected_features))

,Sebelum,Sesudah cleaning
Jumlah baris,581012,581012
Jumlah kolom,55,55
Missing values,0,0
Duplikasi,0,0


Jumlah fitur model setelah feature engineering dan seleksi: 55


## 11. Menyimpan hasil preprocessing untuk notebook Modelling

File `covtype_preprocessed_split.npz` berisi array train/test dan nama fitur. Jalankan notebook ini terlebih dahulu sebelum notebook 04 agar notebook modelling dapat menggunakan hasil preprocessing yang sama.

File hasil ini dibuat dari dataset asli, bukan data sintetis.

In [12]:
output_path = Path("covtype_preprocessed_split.npz")

np.savez_compressed(
    output_path,
    X_train=X_train_selected,
    X_test=X_test_selected,
    y_train=y_train.to_numpy(),
    y_test=y_test.to_numpy(),
    feature_names=np.asarray(selected_features, dtype=str)
)

print(f"Hasil preprocessing tersimpan: {output_path.resolve()}")
print(f"Ukuran file: {output_path.stat().st_size / (1024**2):.2f} MB")

Hasil preprocessing tersimpan: C:\Users\parsey\Downloads\Covertype_4_Phase_Notebooks\covtype_preprocessed_split.npz
Ukuran file: 11.70 MB


## 12. Kesimpulan preprocessing

Tuliskan kesimpulan berdasarkan hasil aktual:

- Apakah data memerlukan imputasi?
- Apakah terdapat baris duplikat?
- Bagaimana keputusan terhadap outlier dan mengapa?
- Mengapa `Aspect` ditransformasikan menjadi sinus dan cosinus?
- Fitur apa yang dihapus oleh `VarianceThreshold`?
- Mengapa scaling tidak diterapkan untuk model berbasis pohon?
- Apakah proporsi kelas pada train dan test relatif terjaga?

**Catatan reproduksibilitas:** gunakan `random_state=42` dan file `covtype_preprocessed_split.npz` yang dihasilkan notebook ini untuk menjaga konsistensi dengan notebook modelling.